In [14]:
import pandas as pd

df = pd.read_csv("Datos/planets.csv")
df.head()

,rowid,pl_hostname,pl_letter,pl_discmethod,pl_pnum,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,...,st_masserr1,st_masserr2,st_masslim,st_massblend,st_rad,st_raderr1,st_raderr2,st_radlim,st_radblend,rowupdate
0,1,11 Com,b,Radial Velocity,1,326.03,0.32,-0.32,0.0,1.290,...,0.30,-0.30,0.0,0.0,19.00,2.00,-2.00,0.0,0.0,2014-05-14
1,2,11 UMi,b,Radial Velocity,1,516.22,3.25,-3.25,0.0,1.540,...,0.25,-0.25,0.0,0.0,24.08,1.84,-1.84,0.0,0.0,2014-05-14
2,3,14 And,b,Radial Velocity,1,185.84,0.23,-0.23,0.0,0.830,...,0.10,-0.20,0.0,0.0,11.00,1.00,-1.00,0.0,0.0,2014-05-14
3,4,14 Her,b,Radial Velocity,1,1773.40,2.50,-2.50,0.0,2.770,...,0.05,-0.05,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2014-05-14
4,5,16 Cyg B,b,Radial Velocity,1,798.50,1.00,-1.00,0.0,1.681,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2015-09-10


## Fase 2: Limpieza de Datos (Data Cleaning)

- **Filtro**: Seleccionar solo las columnas relevantes para el análisis, eliminando columnas de error, flags técnicos y redundancias.
- **Nulos**: Identificar columnas vacías. Decidir si se borran o imputan (justificándolo en un comentario).
- **Duplicados**: Verificar registros repetidos por nombre de estrella y letra de planeta.
- **Outliers**: Revisar valores físicamente imposibles en las variables numéricas.

In [15]:
# ============================================================
# FASE 2: Limpieza de Datos (Data Cleaning)
# ============================================================

print(f"El dataset tiene {len(df)} filas y {df.shape[1]} columnas")
print(f"\nTipos de datos:")
print(df.dtypes.value_counts())
df.head()

El dataset tiene 3372 filas y 67 columnas

Tipos de datos:
float64    53
object      8
int64       6
Name: count, dtype: int64


,rowid,pl_hostname,pl_letter,pl_discmethod,pl_pnum,pl_orbper,pl_orbpererr1,pl_orbpererr2,pl_orbperlim,pl_orbsmax,...,st_masserr1,st_masserr2,st_masslim,st_massblend,st_rad,st_raderr1,st_raderr2,st_radlim,st_radblend,rowupdate
0,1,11 Com,b,Radial Velocity,1,326.03,0.32,-0.32,0.0,1.290,...,0.30,-0.30,0.0,0.0,19.00,2.00,-2.00,0.0,0.0,2014-05-14
1,2,11 UMi,b,Radial Velocity,1,516.22,3.25,-3.25,0.0,1.540,...,0.25,-0.25,0.0,0.0,24.08,1.84,-1.84,0.0,0.0,2014-05-14
2,3,14 And,b,Radial Velocity,1,185.84,0.23,-0.23,0.0,0.830,...,0.10,-0.20,0.0,0.0,11.00,1.00,-1.00,0.0,0.0,2014-05-14
3,4,14 Her,b,Radial Velocity,1,1773.40,2.50,-2.50,0.0,2.770,...,0.05,-0.05,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2014-05-14
4,5,16 Cyg B,b,Radial Velocity,1,798.50,1.00,-1.00,0.0,1.681,...,NaN,NaN,0.0,0.0,NaN,NaN,NaN,NaN,0.0,2015-09-10


In [16]:
# --- 2. Nulos: Identificar columnas con valores faltantes ---

print("Valores nulos por columna:")
print(df.isnull().sum())

Valores nulos por columna:
rowid              0
pl_hostname        0
pl_letter          0
pl_discmethod      0
pl_pnum            0
                ... 
st_raderr1       417
st_raderr2       494
st_radlim        358
st_radblend      187
rowupdate          0
Length: 67, dtype: int64


In [17]:
# Hacemos una copia para no modificar el original
df_clean = df.copy()

# Eliminamos columnas con más del 80% de nulos porque no sirven para el análisis
pct_nulos = df_clean.isnull().mean()
cols_muchos_nulos = pct_nulos[pct_nulos > 0.80].index.tolist()
print(f"Columnas con >80% nulos (se eliminan): {cols_muchos_nulos}")
df_clean.drop(columns=cols_muchos_nulos, inplace=True)

# Eliminamos columnas de error (*err1, *err2) porque son márgenes de incertidumbre
cols_error = [col for col in df_clean.columns if col.endswith('err1') or col.endswith('err2')]
print(f"\nColumnas de error eliminadas: {cols_error}")
df_clean.drop(columns=cols_error, inplace=True)

# Eliminamos columnas de flags técnicos (*lim, *blend) que no aportan al análisis
cols_flags = [col for col in df_clean.columns if col.endswith('lim') or col.endswith('blend')]
print(f"\nColumnas de flags eliminadas: {cols_flags}")
df_clean.drop(columns=cols_flags, inplace=True)

# Eliminamos columnas que no necesitamos (índice, coordenadas repetidas, etc.)
cols_redundantes = ['rowid', 'ra_str', 'dec_str', 'rowupdate', 'pl_nnotes']
cols_redundantes = [col for col in cols_redundantes if col in df_clean.columns]
print(f"\nColumnas redundantes eliminadas: {cols_redundantes}")
df_clean.drop(columns=cols_redundantes, inplace=True)

print(f"\nAhora el dataset tiene: {df_clean.shape[0]} filas y {df_clean.shape[1]} columnas")

# Rellenamos nulos en columnas numéricas con la mediana
cols_numericas = df_clean.select_dtypes(include=['float64', 'int64']).columns.tolist()

for col in cols_numericas:
    nulos = df_clean[col].isnull().sum()
    if nulos > 0:
        mediana = df_clean[col].median()
        df_clean[col] = df_clean[col].fillna(mediana)
        print(f"{col}: {nulos} nulos rellenados con mediana ({mediana})")

# Rellenamos nulos en columnas de texto con la moda (valor más frecuente)
cols_texto = df_clean.select_dtypes(include='object').columns

for col in cols_texto:
    nulos = df_clean[col].isnull().sum()
    if nulos > 0:
        moda = df_clean[col].mode()[0]
        df_clean[col] = df_clean[col].fillna(moda)
        print(f"{col}: {nulos} nulos rellenados con moda ('{moda}')")

print("\nVerificamos que ya no hay nulos:")
print(df_clean.isnull().sum())

Columnas con >80% nulos (se eliminan): ['pl_orbincl', 'pl_orbinclerr1', 'pl_orbinclerr2', 'pl_orbincllim', 'pl_dens', 'pl_denserr1', 'pl_denserr2', 'pl_denslim', 'st_optmagerr']

Columnas de error eliminadas: ['pl_orbpererr1', 'pl_orbpererr2', 'pl_orbsmaxerr1', 'pl_orbsmaxerr2', 'pl_orbeccenerr1', 'pl_orbeccenerr2', 'pl_bmassjerr1', 'pl_bmassjerr2', 'pl_radjerr1', 'pl_radjerr2', 'st_disterr1', 'st_disterr2', 'st_tefferr1', 'st_tefferr2', 'st_masserr1', 'st_masserr2', 'st_raderr1', 'st_raderr2']

Columnas de flags eliminadas: ['pl_orbperlim', 'pl_orbsmaxlim', 'pl_orbeccenlim', 'pl_bmassjlim', 'pl_radjlim', 'st_distlim', 'st_optmaglim', 'st_optmagblend', 'st_tefflim', 'st_teffblend', 'st_masslim', 'st_massblend', 'st_radlim', 'st_radblend']

Columnas redundantes eliminadas: ['rowid', 'ra_str', 'dec_str', 'rowupdate', 'pl_nnotes']

Ahora el dataset tiene: 3372 filas y 21 columnas
pl_orbper: 67 nulos rellenados con mediana (12.794585)
pl_orbsmax: 1475 nulos rellenados con mediana (0.122)
p

In [18]:
# --- 3. Duplicados: Verificar registros repetidos ---

# Duplicados completos
dup_completos = df_clean.duplicated(keep=False).sum()
print(f"Duplicados completos: {dup_completos}")

# Duplicados por nombre de estrella + letra del planeta
dup_parciales = df_clean.duplicated(subset=['pl_hostname', 'pl_letter'], keep=False).sum()
print(f"Duplicados por pl_hostname + pl_letter: {dup_parciales}")

# Tercero: duplicados por combinaciones de variables clave
cols_clave = ['pl_hostname', 'pl_letter', 'pl_discmethod', 'pl_orbper', 'pl_bmassj', 'pl_radj',
              'st_teff', 'st_mass', 'st_rad']
dup_clave = df_clean.duplicated(subset=cols_clave, keep=False).sum()
print(f"Duplicados por variables clave ({len(cols_clave)} columnas): {dup_clave}")

if dup_completos > 0 or dup_parciales > 0 or dup_clave > 0:
    # Mostrar los duplicados encontrados con todas las columnas
    mascara = df_clean.duplicated(keep=False)
    print(f"\nEjemplos de registros duplicados (todas las columnas):")
    display(df_clean[mascara].sort_values(['pl_hostname', 'pl_letter']).head(10))
    
    # Eliminar duplicados completos, manteniendo el primer registro
    antes = len(df_clean)
    df_clean.drop_duplicates(inplace=True)
    despues = len(df_clean)
    print(f"\nRegistros antes: {antes}")
    print(f"Registros después: {despues}")
else:
    print("\nNo se encontraron duplicados.")

Duplicados completos: 0
Duplicados por pl_hostname + pl_letter: 0
Duplicados por variables clave (9 columnas): 0

No se encontraron duplicados.


In [19]:
# --- 4. Outliers: Detectar valores físicamente imposibles ---

# Excluimos pl_bmassj, pl_radj y pl_orbeccen del análisis de outliers porque
# tienen demasiados valores imputados (65%, 20% y 71% respectivamente),
# lo que distorsiona el IQR y el capping. Para el feature engineering y
# la visualización usamos los datos originales (df) de estas columnas.
cols_analisis = ['pl_orbper', 'pl_orbsmax',
                 'st_dist', 'st_optmag', 'st_teff', 'st_mass', 'st_rad']

# Primero vemos cuántos outliers hay
resumen = []
for col in cols_analisis:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    fuera = ((df_clean[col] < limite_inf) | (df_clean[col] > limite_sup)).sum()
    resumen.append({'Columna': col, 'Min': df_clean[col].min(), 'Max': df_clean[col].max(),
                    'Outliers': fuera})

print("ANTES de corregir outliers:")
display(pd.DataFrame(resumen))

# Aplicamos capping: los valores fuera de rango se ajustan al límite
for col in cols_analisis:
    Q1 = df_clean[col].quantile(0.25)
    Q3 = df_clean[col].quantile(0.75)
    IQR = Q3 - Q1
    if IQR == 0:
        print(f"{col}: IQR = 0 (datos concentrados en la mediana), NO se aplica capping")
        continue
    limite_inf = Q1 - 1.5 * IQR
    limite_sup = Q3 + 1.5 * IQR
    df_clean.loc[df_clean[col] > limite_sup, col] = limite_sup
    df_clean.loc[df_clean[col] < limite_inf, col] = limite_inf

# Verificamos que se corrigieron
resumen_despues = []
for col in cols_analisis:
    resumen_despues.append({'Columna': col, 'Min': df_clean[col].min(), 'Max': df_clean[col].max()})

print("\nDESPUÉS de corregir outliers:")
display(pd.DataFrame(resumen_despues))

ANTES de corregir outliers:


,Columna,Min,Max,Outliers
0,pl_orbper,0.090706,7300000.00,582
1,pl_orbsmax,0.004400,2500.00,790
2,st_dist,3.210000,8500.00,173
3,st_optmag,0.850000,20.15,302
4,st_teff,575.000000,57000.00,176
5,st_mass,0.020000,3.90,268
6,st_rad,0.040000,51.10,278



DESPUÉS de corregir outliers:


,Columna,Min,Max
0,pl_orbper,0.090706,100.192488
1,pl_orbsmax,0.032888,0.219188
2,st_dist,3.210000,1324.250000
3,st_optmag,7.810250,19.328250
4,st_teff,4021.125000,7056.125000
5,st_mass,0.490000,1.450000
6,st_rad,0.290000,1.730000
